### Oriented Feature Engineering for Exploratory Data Analysis

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
import os

In [2]:
from pathlib import Path

load_dotenv()

DATA_PATH_CLEANED = os.getenv("DATA_PATH_CLEANED")

pd.set_option('display.max_columns', None)
file_path = Path(DATA_PATH_CLEANED) / "cleaned_crime_data.parquet" # type: ignore
df = pd.read_parquet(file_path)

### Extracting new columns from Time 

In [3]:
print(f"Missing Values in date_occurrance: {df['date_occurrence'].isnull().sum()}")
print(f"Missing Values in date_reported: {df['reported_date'].isnull().sum()}")
df['reporting_delay_days'] = (df['reported_date'] - df['date_occurrence']).dt.days

df['hour_occurrence'].head()
df['minute_occurrence'].head()
# Already extracted hour and minute 
"""
    - year
    - month
    - day_of_week
    - quarter
    - is_weekend
    - time_period
"""

df['year_occurrence'] = df['date_occurrence'].dt.year
df['month_occurrence'] = df['date_occurrence'].dt.month
df['day_of_week_num'] = df['date_occurrence'].dt.dayofweek  # 0=Monday, 6=Sunday
df['is_weekend_occurrence'] = (df['date_occurrence'].dt.dayofweek >= 5).astype(int)
df['quarter_occurrence'] = df['date_occurrence'].dt.quarter
df['is_night_occurrence'] = (df['hour_occurrence'] < 6).astype(int)

Missing Values in date_occurrance: 0
Missing Values in date_reported: 0


In [4]:
def categorize_time(hour):
    if 0 <= hour < 6:
        return 'Late Night'
    elif 6 <= hour < 12:
        return 'Morning'
    elif 12 <= hour < 18:
        return 'Afternoon'
    else:
        return 'Evening'
df['time_period_occurrence'] = df['hour_occurrence'].apply(categorize_time)

In [ ]:
"""
 Spatial Features
- Geo based grouping
    - crime by area
    - lat_bin
    - lon_bin
- Victim Grouping
age_group = (0-18, 19-30, 31-50, 51+)
- Weapon Simplification: 
weapon_flag = 1 if weapon exists else 0

"""

# crime per area
area_counts = df['area_name'].value_counts()
area_crime_share = area_counts / len(df)
df['area_crime_share'] = df['area_name'].map(area_crime_share)
area_crime_share

area_name
Central        0.069191
77th Street    0.061470
Pacific        0.059105
Southwest      0.057239
Hollywood      0.052005
N Hollywood    0.050923
Olympic        0.049864
Southeast      0.049649
Newton         0.048927
Wilshire       0.047974
Rampart        0.046602
West LA        0.045598
Northeast      0.042812
Van Nuys       0.042727
West Valley    0.042035
Devonshire     0.041522
Topanga        0.041267
Harbor         0.041158
Mission        0.040092
Hollenbeck     0.036893
Foothill       0.032946
Name: count, dtype: float64

In [6]:
grid = 0.005  # higher resolution

df['lat_bin'] = (df['latitude'] // grid) * grid
df['lon_bin'] = (df['longitude'] // grid) * grid

# count crimes by location
crime_count_by_location = df.groupby(["lat_bin", "lon_bin"]).size().reset_index(name="crime_count")

df = df.merge(crime_count_by_location, on=['lat_bin', 'lon_bin'], how='left')

In [7]:
# victim age grouping
def categorize_age(age):
    if pd.isna(age) or age == 0:
        return "Unknown"
    elif age <= 18:
        return "0-18"
    elif age <= 30:
        return "19-30"
    elif age <= 50:
        return "31-50"
    else:
        return "51+"
df['age_group'] = df['victim_age'].apply(categorize_age)

In [8]:
# weapon flag
df['weapon_flag'] = (
    df['weapon_description'].notna() &
    (df['weapon_description'].str.upper() != 'UNKNOWN')
).astype(int)

In [9]:
df['crime_code_description'].unique()

<StringArray>
[                                       'theft of identity',
           'assault with deadly weapon, aggravated assault',
      'theft from motor vehicle - grand ($950.01 and over)',
          'theft from motor vehicle - petty ($950 & under)',
 'crm agnst chld (13 or under) (14-15 & susp 10 yrs older)',
                                         'vehicle - stolen',
                                                 'burglary',
                                    'burglary from vehicle',
                       'theft plain - petty ($950 & under)',
                        'intimate partner - simple assault',
 ...
 'beastiality, crime against nature sexual asslt with anim',
                                                   'bigamy',
                                      'failure to disperse',
       'firearms emergency protective order (firearms epo)',
             'incest (sexual acts between blood relatives)',
                           'blocking door induction center',
     

In [10]:
def categorize_crime(desc):
    if pd.isna(desc):
        return "Other"

    desc = desc.lower()

    if any(x in desc for x in ['homicide', 'murder']):
        return 'Violent Crime'

    if any(x in desc for x in ['assault', 'battery', 'robbery']):
        return 'Violent Crime'

    if any(x in desc for x in ['burglary', 'theft', 'larceny']):
        return 'Property Crime'

    if any(x in desc for x in ['vehicle', 'motor vehicle', 'auto theft']):
        return 'Vehicle Crime'

    if any(x in desc for x in ['sexual', 'rape', 'incest']):
        return 'Sexual Crime'

    if any(x in desc for x in ['fraud', 'identity', 'forgery', 'embezzlement']):
        return 'Financial Crime'

    if any(x in desc for x in ['drug', 'narcotic', 'controlled substance']):
        return 'Drug Crime'

    if any(x in desc for x in ['riot', 'disorder', 'disperse']):
        return 'Public Order'

    if any(x in desc for x in ['weapon', 'firearm', 'gun', 'knife']):
        return 'Weapon Crime'

    return 'Other'    
df['crime_category'] = df['crime_code_description'].apply(categorize_crime)

In [11]:
print(df['crime_category'].value_counts())
print(f"Other: {(df['crime_category'] == 'Other').mean():.1%}")

crime_category
Property Crime     410092
Violent Crime      241511
Other              179928
Vehicle Crime      122136
Weapon Crime        36455
Sexual Crime         5941
Financial Crime      3569
Public Order           22
Drug Crime             11
Name: count, dtype: int64
Other: 18.0%


In [12]:
def group_premise(x):
    x = str(x).lower()

    if 'street' in x:
        return 'Public Space'
    elif 'residence' in x:
        return 'Residential'
    elif 'store' in x or 'market' in x:
        return 'Commercial'
    elif 'parking' in x:
        return 'Parking Area'
    else:
        return 'Other'

df['premise_group'] = df['premise_description'].apply(group_premise)

In [13]:
print(df[df['premise_group'] == 'Other']['premise_description'].value_counts().head(20))

premise_description
SINGLE FAMILY DWELLING                          163105
MULTI-UNIT DWELLING (APARTMENT, DUPLEX, ETC)    118582
OTHER BUSINESS                                   47384
SIDEWALK                                         40648
VEHICLE, PASSENGER/TRUCK                         29235
GARAGE/CARPORT                                   19239
DRIVEWAY                                         15999
RESTAURANT/FAST FOOD                             12241
OTHER PREMISE                                     7993
ALLEY                                             7034
PARK/PLAYGROUND                                   6615
YARD (RESIDENTIAL/BUSINESS)                       6151
GAS STATION                                       5722
HOTEL                                             5441
TRANSPORTATION FACILITY (AIRPORT)                 4467
PORCH, RESIDENTIAL                                4347
MINI-MART                                         3920
BANK                                         

In [14]:
output_path = Path(DATA_PATH_CLEANED) / "feature_engineered_crime_data.parquet" # type: ignore
df.to_parquet(output_path, index=False)
